## Parte 2. Revisión del Dataset

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv("ventas_ecommerce_limpio.csv")
df.head()

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta
0,1001,2026-07-01,Ana Lopez,Mouse,Accesorios,1,250.0,Efectivo,Cuernavaca,250.0
1,1002,2026-07-01,Luis Perez,Teclado,Accesorios,1,650.0,Tarjeta,Jiutepec,650.0
2,1003,2026-07-02,Sofia Ruiz,Audifonos,Accesorios,1,900.0,Tarjeta,Temixco,900.0
3,1004,2026-07-02,Pedro Mata,Webcam,Accesorios,1,800.0,Efectivo,Cuernavaca,800.0
4,1005,2026-07-03,Laura Diaz,Cable HDMI,Accesorios,2,180.0,Efectivo,Jiutepec,360.0


In [2]:
df.columns

Index(['id_venta', 'fecha', 'cliente', 'producto', 'categoria', 'cantidad',
       'precio_unitario', 'metodo_pago', 'ciudad', 'total_venta'],
      dtype='object')

In [3]:
df.shape

(60, 10)

In [4]:
df.isnull().sum()

id_venta           0
fecha              0
cliente            0
producto           0
categoria          0
cantidad           0
precio_unitario    0
metodo_pago        0
ciudad             0
total_venta        0
dtype: int64

In [5]:
'total_venta' in df.columns

True

In [6]:
df["total_calculado"] = df["cantidad"] * df["precio_unitario"]
inconsistencias = df[df["total_venta"] != df["total_calculado"]]
inconsistencias

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta,total_calculado


In [7]:
df = df.drop(columns=["total_calculado"])

## Parte 3. Variable Objetivo

In [8]:
def clasificar_venta(total):
    if total >= 1000:
        return 1
    else:
        return 0

df["venta_alta"] = df["total_venta"].apply(clasificar_venta)
df["venta_alta"].value_counts()

venta_alta
1    40
0    20
Name: count, dtype: int64

**Pregunta obligatoria:**

Por que venta_alta es la variable objetivo? por que es la variable a la cual se intenta llegar o predecir

## Parte 4. Variables de Entrada

In [9]:
X = df[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]
y = df["venta_alta"]

X = pd.get_dummies(X)
columnas_modelo = X.columns.tolist()
columnas_modelo

['cantidad',
 'precio_unitario',
 'categoria_Accesorios',
 'categoria_Electronica',
 'categoria_Muebles',
 'metodo_pago_Efectivo',
 'metodo_pago_Tarjeta',
 'metodo_pago_Transferencia',
 'ciudad_Cuernavaca',
 'ciudad_Emiliano Zapata',
 'ciudad_Jiutepec',
 'ciudad_Temixco']

In [10]:
X.head()

,cantidad,precio_unitario,categoria_Accesorios,categoria_Electronica,categoria_Muebles,metodo_pago_Efectivo,metodo_pago_Tarjeta,metodo_pago_Transferencia,ciudad_Cuernavaca,ciudad_Emiliano Zapata,ciudad_Jiutepec,ciudad_Temixco
0,1,250.0,True,False,False,True,False,False,True,False,False,False
1,1,650.0,True,False,False,False,True,False,False,False,True,False
2,1,900.0,True,False,False,False,True,False,False,False,False,True
3,1,800.0,True,False,False,True,False,False,True,False,False,False
4,2,180.0,True,False,False,True,False,False,False,False,True,False


**Pregunta obligatoria:**

Por que no se debe usar total_venta como variable de entrada si venta_alta se creo a partir de total_venta? Porque total_venta fue la que se usó para calcular venta_alta, entonces si la meto como variable de entrada el modelo básicamente ya sabría la respuesta de antemano, no estaría aprendiendo nada real.

## Parte 5. Entrenamiento y Evaluación

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

((48, 12), (12, 12))

In [12]:
modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [13]:
predicciones = modelo.predict(X_test)

exactitud = accuracy_score(y_test, predicciones)
print("Exactitud:", exactitud)

Exactitud: 1.0


In [14]:
matriz = confusion_matrix(y_test, predicciones)
print(matriz)

[[4 0]
 [0 8]]


In [15]:
resultados_prueba = pd.DataFrame({
    "valor_real": y_test,
    "prediccion": predicciones
})

resultados_prueba["coincide"] = resultados_prueba["valor_real"] == resultados_prueba["prediccion"]
resultados_prueba

,valor_real,prediccion,coincide
0,0,0,True
5,0,0,True
36,1,1,True
45,1,1,True
13,0,0,True
54,1,1,True
33,1,1,True
48,1,1,True
12,0,0,True
57,1,1,True


In [16]:
resultados_prueba["coincide"].value_counts()

coincide
True    12
Name: count, dtype: int64

In [17]:
aciertos = resultados_prueba[resultados_prueba["coincide"] == True]
errores = resultados_prueba[resultados_prueba["coincide"] == False]

print("Aciertos:", len(aciertos))
print("Errores:", len(errores))

Aciertos: 12
Errores: 0


**Preguntas obligatorias:**

1. Cual fue la exactitud? 1.0
2. Cuantos aciertos tuvo el modelo? 12
3. Cuantos errores tuvo el modelo? 0
4. Que indica la matriz de confusion? Que no hubo confusiones, acerto las 4 ventas no altas y las 8 altas del test
5. Una buena exactitud significa que el modelo ya es perfecto? Explica. No, porque el dataset es chico y con otros datos podria fallar mas

## Parte 6. Guardar Modelo y Columnas

In [18]:
joblib.dump(modelo, "modelo_examen_venta_alta.pkl")
joblib.dump(columnas_modelo, "columnas_examen_modelo.pkl")
print("Archivos guardados correctamente")

Archivos guardados correctamente


**Preguntas obligatorias:**

1. Para que sirve guardar el modelo? Para poder reutilizarlo despues sin tener que volver a entrenarlo cada vez.
2. Para que sirve guardar las columnas del entrenamiento? Permite reconstruir esa estructura al preparar datos nuevos.
3. Que problema puede aparecer si no guardas las columnas? Se pueden generar columnas distintas, el modelo recibiria datos con una forma distinta a la esperada y fallaria o generaria predicciones incorrectas.

## Parte 7. Ventas Nuevas

In [19]:
nuevas = pd.DataFrame({
    "id_venta": [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012],
    "fecha": [
        "2026-08-07", "2026-08-07", "2026-08-08", "2026-08-08",
        "2026-08-09", "2026-08-09", "2026-08-10", "2026-08-10",
        "2026-08-11", "2026-08-11", "2026-08-12", "2026-08-12"
    ],
    "cliente": [
        "Andres Garcia", "Orlando Ruiz", "Cesar Emilio", "Sebastian Jimenez",
        "Citlalli Fausto", "Jocelyn Jimenez", "Sashenka Ureña", "Enrique Jimenez",
        "Eliel Rodriguez", "Juan Diaz", "Erick Teja", "Astrid Valeria"
    ],
    "producto": [
        "Laptop Gamer", "Refrigerador", "Sofa 3 plazas", "Smart TV 55'",
        "Mouse basico", "Cable USB", "Audifonos economicos", "Playera basica",
        "Silla de oficina", "Impresora multifuncion", "Tenis deportivos", "Bolsa de mano"
    ],
    "categoria": [
        "Electronica", "Muebles", "Muebles", "Electronica",
        "Accesorios", "Accesorios", "Accesorios", "Ropa",
        "Muebles", "Electronica", "Ropa", "Ropa"
    ],
    "cantidad": [1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1],
    "precio_unitario": [18000, 9500, 7200, 8300, 200, 90, 350, 250, 980, 1050, 1200, 250],
    "metodo_pago": [
        "Tarjeta", "Transferencia", "Tarjeta", "Efectivo",
        "Efectivo", "Efectivo", "Tarjeta", "Efectivo",
        "Tarjeta", "Transferencia", "Tarjeta", "Efectivo"
    ],
    "ciudad": [
        "Cuernavaca", "Temixco", "Jiutepec", "Emiliano Zapata",
        "Cuernavaca", "Jiutepec", "Temixco", "Cuautla",
        "Emiliano Zapata", "Cuernavaca", "Cuautla", "Cuautla"
    ]
})

nuevas.to_csv("examen_ventas_nuevas.csv", index=False)
nuevas

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad
0,2001,2026-08-07,Andres Garcia,Laptop Gamer,Electronica,1,18000,Tarjeta,Cuernavaca
1,2002,2026-08-07,Orlando Ruiz,Refrigerador,Muebles,1,9500,Transferencia,Temixco
2,2003,2026-08-08,Cesar Emilio,Sofa 3 plazas,Muebles,1,7200,Tarjeta,Jiutepec
3,2004,2026-08-08,Sebastian Jimenez,Smart TV 55',Electronica,1,8300,Efectivo,Emiliano Zapata
4,2005,2026-08-09,Citlalli Fausto,Mouse basico,Accesorios,1,200,Efectivo,Cuernavaca
5,2006,2026-08-09,Jocelyn Jimenez,Cable USB,Accesorios,2,90,Efectivo,Jiutepec
6,2007,2026-08-10,Sashenka Ureña,Audifonos economicos,Accesorios,1,350,Tarjeta,Temixco
7,2008,2026-08-10,Enrique Jimenez,Playera basica,Ropa,2,250,Efectivo,Cuautla
8,2009,2026-08-11,Eliel Rodriguez,Silla de oficina,Muebles,1,980,Tarjeta,Emiliano Zapata
9,2010,2026-08-11,Juan Diaz,Impresora multifuncion,Electronica,1,1050,Transferencia,Cuernavaca


## Parte 8. Cargar Modelo y Predecir

In [20]:
modelo_cargado = joblib.load("modelo_examen_venta_alta.pkl")
columnas_modelo = joblib.load("columnas_examen_modelo.pkl")

nuevas = pd.read_csv("examen_ventas_nuevas.csv")

X_nuevas = nuevas[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]
X_nuevas = pd.get_dummies(X_nuevas)
X_nuevas = X_nuevas.reindex(columns=columnas_modelo, fill_value=0)
X_nuevas.head()

,cantidad,precio_unitario,categoria_Accesorios,categoria_Electronica,categoria_Muebles,metodo_pago_Efectivo,metodo_pago_Tarjeta,metodo_pago_Transferencia,ciudad_Cuernavaca,ciudad_Emiliano Zapata,ciudad_Jiutepec,ciudad_Temixco
0,1,18000,False,True,False,False,True,False,True,False,False,False
1,1,9500,False,False,True,False,False,True,False,False,False,True
2,1,7200,False,False,True,False,True,False,False,False,True,False
3,1,8300,False,True,False,True,False,False,False,True,False,False
4,1,200,True,False,False,True,False,False,True,False,False,False


In [21]:
nuevas["prediccion_venta_alta"] = modelo_cargado.predict(X_nuevas)

nuevas["interpretacion_prediccion"] = nuevas["prediccion_venta_alta"].map({
    0: "Venta no alta",
    1: "Venta alta"
})

nuevas[["id_venta", "producto", "categoria", "ciudad", "prediccion_venta_alta", "interpretacion_prediccion"]]

,id_venta,producto,categoria,ciudad,prediccion_venta_alta,interpretacion_prediccion
0,2001,Laptop Gamer,Electronica,Cuernavaca,1,Venta alta
1,2002,Refrigerador,Muebles,Temixco,1,Venta alta
2,2003,Sofa 3 plazas,Muebles,Jiutepec,1,Venta alta
3,2004,Smart TV 55',Electronica,Emiliano Zapata,1,Venta alta
4,2005,Mouse basico,Accesorios,Cuernavaca,0,Venta no alta
5,2006,Cable USB,Accesorios,Jiutepec,0,Venta no alta
6,2007,Audifonos economicos,Accesorios,Temixco,0,Venta no alta
7,2008,Playera basica,Ropa,Cuautla,1,Venta alta
8,2009,Silla de oficina,Muebles,Emiliano Zapata,1,Venta alta
9,2010,Impresora multifuncion,Electronica,Cuernavaca,1,Venta alta


In [22]:
nuevas.to_csv("examen_predicciones.csv", index=False)

**Pregunta obligatoria:**

Que podria pasar si no usas reindex antes de predecir? Si no uso reindex, las columnas de los datos nuevos pueden no coincidir con las del entrenamiento (por ejemplo faltaria alguna ciudad o categoria), y el modelo tronaria o predecidiria mal.

## Parte 9. Auditoría del Modelo

In [23]:
nuevas["total_estimado"] = nuevas["cantidad"] * nuevas["precio_unitario"]
nuevas["venta_alta_real_estimada"] = nuevas["total_estimado"].apply(lambda x: 1 if x >= 1000 else 0)

nuevas["coincide"] = nuevas["prediccion_venta_alta"] == nuevas["venta_alta_real_estimada"]
nuevas["coincide"].value_counts()

coincide
True     9
False    3
Name: count, dtype: int64

In [24]:
errores = nuevas[nuevas["coincide"] == False]
errores

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide
7,2008,2026-08-10,Enrique Jimenez,Playera basica,Ropa,2,250,Efectivo,Cuautla,1,Venta alta,500,0,False
8,2009,2026-08-11,Eliel Rodriguez,Silla de oficina,Muebles,1,980,Tarjeta,Emiliano Zapata,1,Venta alta,980,0,False
11,2012,2026-08-12,Astrid Valeria,Bolsa de mano,Ropa,1,250,Efectivo,Cuautla,1,Venta alta,250,0,False


In [25]:
nuevas["distancia_a_1000"] = (nuevas["total_estimado"] - 1000).abs()
cerca_limite = nuevas[nuevas["distancia_a_1000"] <= 200]
cerca_limite[["id_venta", "producto", "total_estimado", "prediccion_venta_alta", "coincide"]]

,id_venta,producto,total_estimado,prediccion_venta_alta,coincide
8,2009,Silla de oficina,980,1,False
9,2010,Impresora multifuncion,1050,1,True
10,2011,Tenis deportivos,1200,1,True


In [26]:
nuevas.to_csv("examen_predicciones.csv", index=False)
print("Archivo actualizado con columnas de auditoria")

Archivo actualizado con columnas de auditoria


**Preguntas obligatorias:**

1. Cuantas ventas nuevas evaluaste? 12
2. Cuantas fueron predichas como venta alta? 9
3. Cuantas fueron predichas como venta no alta? 3
4. Cuantas coincidieron con la regla manual? 9
5. Cuantas no coincidieron? 3
6. Que ventas no coincidieron? 2008, 2009 y 2012
7. Los errores estuvieron cerca del limite de 1000? Si, la 2009 quedo a solo 20 peso del limite
8. Que paso con la categoria nueva? Como el modelo nunca vio "ropa", no tenia columna para eso y fallo 2 ventas de esas
9. Que paso con la ciudad nueva? Igual que con la ropa, como cuautla es ciudad nueva, no tenia columna y el modelo la evaluo sin esa informacion
    
**Resumen:**
Nombre: Jimenez Ureña Angel Sebastian 
Grupo: 9A
Materia: Extraccion de conocimiento en base de datos 
Exactitud obtenida: 1.0
Ventas nuevas evaluadas: 12
Coincidencias: 9
Errores: 3
Conclusion breve: El modelo acerto todo en el test y 9 de 12 en las ventas nuevas. Los errores fueron
en la categoría "Ropa" que era nueva y en una venta muy cerca del límite de 1000.